# LoRA(Low-Rank Adaptation)

LoRA는 큰 선형층의 원래 가중치 `W`를 고정하고, 가중치 변화량 `ΔW`만 두 작은 저랭크 행렬의 곱으로 학습하는 PEFT 방법이다. 전체 가중치 대신 작은 adapter만 저장·교체할 수 있어 제한된 GPU에서도 업무별 모델을 파인튜닝할 수 있다.


## LoRA 구조와 수식

선형층 하나는 샘플마다 `d_in`개의 입력값을 받아 `d_out`개의 출력값을 만든다.

원래 가중치 `W`의 shape는 `(d_out, d_in)`이고, 전체 파인튜닝은 이 큰 행렬의 모든 변화량을 학습해야 한다.

LoRA는 업무에 필요한 가중치 변화 `ΔW`가 모든 방향을 독립적으로 바꿀 필요는 없다고 가정한다.

대신 변화량을 표현할 작은 내부 차원 `r`을 정하고, `A`와 `B` 두 행렬만 학습한다.

행렬의 **rank**는 서로 독립적인 정보 방향의 수이며, `ΔW = B @ A`의 rank는 최대 `r`이다.

따라서 `r`은 adapter가 사용할 수 있는 변화 방향의 수를 제한한다.
( == 현재 입력에서 어떤 방향의 변화가 필요한지 핵심 방해을 지정/제한)

`A.T`는 입력 하나를 `r`개의 **adapter 변화 신호**로 바꾼다. 이어서 `B.T`는 그 신호를 기존 선형층과 같은 `d_out`개의 **출력 보정값**으로 바꾼다. 이 과정은 입력을 압축했다가 원래 입력으로 복원하는 것이 아니다. 마지막 결과를 base 출력에 더해야 하므로 adapter도 반드시 `d_out`개의 값을 만들어야 한다.

현재 코드처럼 샘플 3개, 입력 특성 8개, 출력 특성 4개, `r=2`라면 두 경로는 다음과 같이 진행된다.

- base 경로: `X(3, 8) @ W.T(8, 4) → base_output(3, 4)`
- adapter 1단계: `X(3, 8) @ A.T(8, 2) → adapter_hidden(3, 2)`
- adapter 2단계: `adapter_hidden(3, 2) @ B.T(2, 4) → adapter_correction(3, 4)`
- 최종 덧셈: `base_output(3, 4) + adapter_correction(3, 4) → Y(3, 4)`

즉, base 경로는 `8 → 4`, adapter 경로는 `8 → 2 → 4`로 진행한다. 가운데 2차원은 최종 출력이 아니라 적은 파라미터로 변화량을 만들기 위한 작은 작업 공간이다.

$$Y = XW^T + \frac{\alpha}{r}(XA^T)B^T, \qquad \Delta W = BA$$

각 기호의 역할과 shape는 다음과 같다.

- `X: (batch, d_in)`은 선형층에 들어가는 데이터이다. `batch`는 한 번에 처리하는 샘플 수이고, `d_in`은 샘플 하나의 입력값 수이다.
- `W: (d_out, d_in)`은 사전학습 모델이 이미 가진 원래 가중치이며 LoRA 학습 중에는 고정한다. 입력을 행 단위로 저장하므로 코드에서는 `X @ W.T`로 계산한다.
- `A: (r, d_in)`은 입력마다 `r`개의 adapter 변화 신호를 만든다. 코드에서는 `X @ A.T`의 결과가 `(batch, r)`이 된다.
- `B: (d_out, r)`은 `r`개의 변화 신호를 `d_out`개의 출력 보정값으로 바꾼다. 코드에서는 `(X @ A.T) @ B.T`의 결과가 `(batch, d_out)`이 된다.
- `ΔW = B @ A: (d_out, d_in)`은 두 adapter 행렬을 하나로 합친 가중치 변화량이다. `W`와 shape가 같으므로 `W + ΔW`를 계산할 수 있다.

```mermaid
flowchart LR
    X["입력 X<br/>(batch, d_in)"] --> BASE["고정 경로<br/>X @ W.T"]
    X --> DOWN["A.T: adapter 변화 신호<br/>d_in → r"]
    DOWN --> UP["B.T: 출력 보정값<br/>r → d_out"]
    UP --> SCALE["alpha / r 적용"]
    BASE --> ADD(("+"))
    SCALE --> ADD
    ADD --> Y["출력 Y<br/>(batch, d_out)"]
```

위쪽 base 경로는 고정되고, 아래쪽 `A → B → scaling` 경로만 학습된다. `A`는 난수로, `B`는 0으로 초기화하므로 시작 시 `B @ A = 0`이 되어 adapter를 붙이기 전과 출력이 같다. 학습 코드에서 `module.parameters()`를 optimizer에 전달하면 등록된 `A`와 `B`가 모두 갱신될 수 있다.

`ΔW` 전체를 직접 학습하면 파라미터가 `d_out × d_in`개 필요하지만, LoRA는 `A`와 `B`를 합쳐 `r × d_in + d_out × r`개만 학습한다. LoRA의 파라미터가 실제로 더 적으려면 `r(d_in + d_out) < d_in × d_out`이어야 한다. 현재 작은 예제는 32개 대신 24개를 학습하며, 실제 LLM에서는 입출력 차원이 수천이고 `r`은 보통 그보다 매우 작아 절감 폭이 훨씬 커진다. `alpha / r`은 학습된 adapter 보정값이 최종 출력에 반영되는 크기를 조절한다.

참고: [LoRA 논문](https://arxiv.org/abs/2106.09685), [Hugging Face PEFT LoRA](https://huggingface.co/docs/peft/developer_guides/lora)


## 행렬로 LoRA 계산하기

입력 특성 8개를 출력 특성 4개로 바꾸는 작은 선형층에서 base 경로 `8 → 4`와 adapter 경로 `8 → 2 → 4`를 각각 계산한다. `adapter_hidden`은 `A`가 만든 2개의 변화 신호이고, `adapter_correction`은 `B`가 만든 4개의 출력 보정값이다. `B`를 0으로 초기화하면 처음에는 보정값이 0이므로 base 출력이 그대로 유지된다.


In [1]:
import torch

torch.manual_seed(42) # 랜덤값 고정

# X: 입력 샘플 3행
X = torch.randn(3, 8) # 입력 3행, 각 3행은 8개의 feature(벡터값)

W = torch.randn(4, 8) # 출력 4행, 입력 feature 8열

base_output = X @ W.T # (3, 4), 3행 입력 각각 4개의 feature로 출력

# r(Rank) : 각 샘플마다 사용할 중간 변환 신호
rank = 2  # 기존: 8 -> 4 ,  r 적용: 8 -> 2 -> 4

# alpha: LoRA가 만든 보정값 (△W)의 영향력을 조절하는 값
alpha = 8

# alpha/rank == 보정값 (△W)에 곱해지는 스케일링 값

# 기존 선형층을 나눔

# A: LoRA가 사용할 핵심 변화 신호를 만들어내는 행렬
A = 0.01 * torch.randn(rank, 8)

# B: 원래 출력 차원으로 변환하는 행렬
B = torch.zeros(4, rank)

# 입력(3,8) @ A.T(8,2) = (3,2)
adapter_hidden = X @ A.T

# A출력(3,2) @ B.T(2,4) = (3,4)
adapter_correction = adapter_hidden @ B.T

# delta_W : 추가 학습에 의한 파라미터 변화량
# == 두 adapter 단계를 W와 같은 변화량 하나로 합친 표현
delta_W = B @ A

lora_output = base_output + (alpha/rank) * adapter_correction

print('base output shape:', base_output.shape)
print('adapter hidden shape:', adapter_hidden.shape)
print('adapter correction shape:', adapter_correction.shape)
print('delta_W shape:', delta_W.shape)
print('LoRA output shape:', lora_output.shape)
print('initial delta max:', delta_W.abs().max().item())

# [사전 학습 모델]
# X(3,8) -> Linear(8,4) -> Output(3, 4)
#             W(4,8)

# [LoRA 적용]
# X(3, 8) -> A(2, 8) -> X@A.T(3, 2) ->  B(4, 2) -> Output(3, 4)
#                                                 (X@A.T) @ B.T

# 사전 학습 모델과, LoRA 적용 모델의 최종 Output 차원 수(shape)가 같음!
# == LoRA는 작은 중간 공간을 거치지만 최종 shape는 기존 선형층과 같다!

# 이때, LoRA는 가중치를 기존 가중치(W) + 추가 학습 가중치(△W)를 이용해서
# 보정된 출력 값을 만든다.

base output shape: torch.Size([3, 4])
adapter hidden shape: torch.Size([3, 2])
adapter correction shape: torch.Size([3, 4])
delta_W shape: torch.Size([4, 8])
LoRA output shape: torch.Size([3, 4])
initial delta max: 0.0


## `nn.Module`로 LoRA 선형층 구현하기

`nn.Module`은 학습 파라미터와 `forward()` 연산을 한 객체로 묶는 PyTorch 기본 클래스이다. `nn.Parameter`로 등록한 `A`, `B`는 `module.parameters()`에 포함되어 optimizer에 전달할 수 있고, `base.weight.requires_grad_(False)`는 원래 가중치의 gradient 계산을 끈다.

생성자의 `in_features`와 `out_features`는 선형층의 입력·출력 차원이고, `rank`는 adapter의 내부 차원이다. `alpha`는 `alpha / rank` scaling을 결정한다. `forward()`는 base 경로와 adapter 경로를 각각 `(batch, out_features)`로 만든 뒤 더한다.


In [2]:
import math
import torch.nn as nn

# 원래 가중치를 고정하고, A와 B만 학습하는 LoRA 선형층 구현
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank=2, alpha=2):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank

        # base는 사전학습 가중치에 해당하며 LoRA 학습 중에는 고정한다.
        self.base = nn.Linear(in_features, out_features, bias=False)

        # 기존 선형층의 가중치 변화 X
        self.base.weight.requires_grad_(False)

        # A는 난수, B는 0으로 초기화하지만 두 행렬은 모두 학습 대상이다.
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        # 기존 가중치 W(4,8)
        base_output = self.base(x)

        # A는 r개의 변화 신호를 만들고 B는 기존 선형층과 같은 출력 보정값으로 바꾼다.
        adapter_output = (x @ self.A.T) @ self.B.T
        return base_output + self.scaling * adapter_output


lora_layer = LoRALinear(8, 4, rank=2, alpha=8)
sample_batch = torch.randn(3, 8)
output = lora_layer(sample_batch)

print('output shape:', output.shape)
print('output:', output)

# 기존 가중치 학습 여부 확인 -> False
print('base weight trainable:', lora_layer.base.weight.requires_grad)

# 선형층 A,B 학습 여부 확인 -> True
# [A]
# == 지정된 rank로 줄어든 차원
# == 추가 학습 내용을 표현할 핵심 변화 신호를 만들어내는 행렬

# [B]
# == 기존 출력에 맞춰 차원 수 보장하는 행렬
print('A trainable:', lora_layer.A.requires_grad)
print('B trainable:', lora_layer.B.requires_grad)

# Full FineTuning 연산 수 : 8 * 4 == 32
# LoRA FineTuning 연산 수 : (8*2) + (2*4) == 24
# --> 연산 수 감소 -> GPU 부하 감소, 학습 속도 증가


output shape: torch.Size([3, 4])
output: tensor([[ 0.7715, -0.5718, -1.2166,  0.0834],
        [ 0.4432, -0.3773,  0.0534,  0.2006],
        [-0.2022, -1.0358, -0.0191, -1.7941]], grad_fn=<AddBackward0>)
base weight trainable: False
A trainable: True
B trainable: True
